# Metajeux-05 : Le temoin d'impossibilite

**Navigation** : [<< Metajeux-04-Chemin-Minimal-Robinson-Goforth](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) | [Index](README.md) | [Metajeux-06-Chemin-Minimal-3x2 >>](Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) | [Python](Metajeux-05-Temoins-Impossibilite-Python.ipynb)

GT-20 construit et verifie un chemin minimal entre deux jeux ; ce carnet attaque la question duale, le **temoin d'impossibilite** — prouver qu'aucun chemin plus court n'existe — en s'appuyant sur la **structure produit** des chambres : `d_chambre(G,H) = d_perm(row_G,row_H) + d_perm(col_G,col_H)`.

**Conventions reprises de GT-20** : `canonique`, `swap_valeurs_adjacentes`, `swap_jeu(cote, k)`, 24 ordres stricts sur 4 cases, 576 chambres, 6 voisins par jeu (3 swaps x {ligne, colonne}).

## 0. Configuration

In [1]:
// === Configuration : GameTheory 20b ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

#[allow(dead_code)]
fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// 24 ordres stricts : BTreeSet => ordre lexicographique (identique au sorted() Python)
let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

// 576 chambres : produit row-major (r parcourt stricts, puis c parcourt stricts)
let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

println!("GameTheory 20b : temoin d'impossibilite certifie sur R-G (point 4)");
println!();
println!("Permutaedre : {} ordres stricts, {} voisins / ordre", stricts.len(), adj_perm(stricts[0]).len());
println!("Chambres : {} sommets, {} voisins / chambre", chambres.len(), adj_chambre(chambres[0]).len());

GameTheory 20b : temoin d'impossibilite certifie sur R-G (point 4)


Permutaedre : 24 ordres stricts, 3 voisins / ordre


Chambres : 576 sommets, 6 voisins / chambre


## 1. La structure produit

Le graphe des chambres est un **produit cartesien** de deux copies du permutaedre `S_4` : chaque chambre `(row, col)` est une paire d'ordres stricts sur 4 cases, et les generateurs ne touchent qu'un **seul cote a la fois** (un swap sur la ligne laisse la colonne intacte, et reciproquement).

**Theoreme cle** : pour tous `G = (row_G, col_G)` et `H = (row_H, col_H)` dans `chambres`, la distance dans le produit est exactement la somme des distances dans chaque facteur : `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

**Demonstration par double inegalite** :

- **Borne inferieure (par comptage des pas de chaque cote)** : soit un chemin de `G` a `H` de longueur `L` dans le produit. Chaque generateur ne touche qu'un seul cote, donc les `L` pas se repartissent en `n_row` pas cote ligne et `n_col` pas cote colonne, avec `L = n_row + n_col`. En ne gardant que les pas cote ligne, on obtient un chemin de `row_G` a `row_H` dans le permutaedre, de longueur `n_row` : donc `n_row >= d_perm(row_G, row_H)`, et symetriquement `n_col >= d_perm(col_G, col_H)`. En sommant : `L >= d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

  *Pourquoi la projection seule ne suffit pas* : projeter le chemin **entier** sur le facteur ligne ne preserve pas la longueur — les pas cote colonne projettent sur du surplace. On en tire `d_perm(row_G, row_H) <= L`, et de meme `d_perm(col_G, col_H) <= L` : cela borne `L` par le **maximum** des deux distances, pas par leur **somme**. C'est le comptage `L = n_row + n_col`, et lui seul, qui donne l'additivite.

- **Borne superieure (par concatenation)** : reciproquement, un chemin optimal dans le facteur ligne (longueur `d_perm(row_G, row_H)`) et un chemin optimal dans le facteur colonne (longueur `d_perm(col_G, col_H)`) peuvent etre concatenes cote par cote pour produire un chemin de longueur `d_perm(row_G, row_H) + d_perm(col_G, col_H)` dans le produit (l'independance des cotes permet l'entrelacement arbitraire). Donc `d_chambre(G, H) <= d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

- **Conclusion** : l'egalite stricte est prouvee pour **toute** paire `(G, H)`. Le produit cartesien preserve exactement la distance metrique — c'est la structure produit du graphe.

In [2]:
// === Section 1.1 : verification exhaustive ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

// 24 BFS sur le permutaedre (un par ordre strict)
let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
// 1 BFS sur le graphe des chambres, depuis IDENTITE
let dist_chambre: HashMap<Jeu, u32> = bfs(IDENTITE, adj_chambre);

// table 24x24 : dtab[i][j] = distance de stricts[i] a stricts[j]
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// verification exhaustive sur les 576 chambres, dans l'ordre row-major
// La ligne dtab[0] est la ligne de distances depuis IDENTITE : on lit donc
// explicitement la table pour comparer le BFS a la somme annoncee.
let mut ecarts: u32 = 0;
for &g in chambres.iter() {
    let (r, c) = g;
    let d_ch = *dist_chambre.get(&g).unwrap();
    let d_calc = dtab[0][sidx[&r]] + dtab[0][sidx[&c]];
    if d_ch != d_calc {
        ecarts += 1;
    }
}

let diam_perm: u32 = dtab.iter().flat_map(|d| d.iter()).copied().max().unwrap();
let diam_ch: u32 = dist_chambre.values().copied().max().unwrap();

println!("Verification exhaustive sur {} chambres : {} ecarts", chambres.len(), ecarts);
println!("Structure produit : {}", if ecarts == 0 { "EXACTE" } else { "NON" });
println!("Diametre permutaedre : {}", diam_perm);
println!("Diametre chambres : {}", diam_ch);
assert!(ecarts == 0);
println!("ASSERTION PASS : structure produit verifiee sur {}/{}", chambres.len(), chambres.len());

Verification exhaustive sur 576 chambres : 0 ecarts


Structure produit : EXACTE


Diametre permutaedre : 6


Diametre chambres : 12


ASSERTION PASS : structure produit verifiee sur 576/576


## 2. Le certificat analytique

Pour un triplet `(G, H, k_max)`, on definit le verdict par comparaison :

- **`IMPOSSIBLE`** si `d_row + d_col > k_max` (la distance minimale theorique depasse la borne, donc aucun chemin de longueur `<= k_max` n'existe) ;
- **`POSSIBLE`** si `d_row + d_col <= k_max` (un chemin de cette longueur existe, par concatenation des chemins optimaux dans chaque facteur).

Le verdict `IMPOSSIBLE` porte un **certificat analytique** : les valeurs `(d_row, d_col, k_max)` qui le justifient. Le verdict `POSSIBLE` porte la meme garantie d'existence, mais ne fournit pas le chemin explicite — celui-ci est produit par le constructeur de GT-20 (point 1 du critere #12205 §5).

**Verification de la table 24x24** : le tableau `dtab` des distances dans le permutaedre doit etre symetrique et triangulaire pour que le certificat soit coherent.

In [3]:
// === Section 2.1 : precalcul 24x24 ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

// 24 BFS sur le permutaedre -> table 24x24
let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// symetrie : boucle24x24 sur les paires (i, j)
let mut non_sym: u32 = 0;
for i in 0..stricts.len() {
    for j in 0..stricts.len() {
        if dtab[i][j] != dtab[j][i] {
            non_sym += 1;
        }
    }
}
println!("Symetrie {}x{} : {} asymetries", stricts.len(), stricts.len(), non_sym);

// inegalite triangulaire : triple boucle 24 x 24 x 24
let mut violations: u32 = 0;
for i in 0..stricts.len() {
    for j in 0..stricts.len() {
        for k in 0..stricts.len() {
            if dtab[i][k] > dtab[i][j] + dtab[j][k] {
                violations += 1;
            }
        }
    }
}
let n = stricts.len() as u64;
println!(
    "Inegalite triangulaire : {} violations sur {} triplets",
    violations,
    n * n * n
);
assert!(violations == 0);
println!("ASSERTION PASS : table 24x24 symetrique et triangulaire");

Symetrie 24x24 : 0 asymetries


Inegalite triangulaire : 0 violations sur 13824 triplets


ASSERTION PASS : table 24x24 symetrique et triangulaire


## 3. Le temoin en action

Trois cas fondateurs illustrent les regimes du certificat :

- **Cas A** : antipode `IDENTITE -> RENVERS`, `k_max=5` -> verdict `IMPOSSIBLE` (distance 12 > 5) ;
- **Cas B** : `IDENTITE` vers une chambre a `d_row=0, d_col=4`, `k_max=3` -> verdict `IMPOSSIBLE` (distance 4 > 3) ;
- **Cas C** : frontiere `d_row=1, d_col=0`, `k_max=1` -> verdict `POSSIBLE` (distance 1 <= 1).

L'exemple ci-dessous montre comment le verdict est obtenu a partir de la table `dtab`. Le retour est un tuple `(&'static str, u32, u32, u32, String)` — `verdict`, `d_row`, `d_col`, `d_min`, `preuve` — ou `preuve` est la chaine qui materialise l'inegalite ; `k_max` reste un argument de la fonction et n'est pas renvoyé separement.

In [4]:
// === Section 3.1 : certifier_impossibilite ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
const RENVERS: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
const H_B: Jeu = ([1, 2, 3, 4], [3, 4, 1, 2]);
const H_C: Jeu = ([1, 2, 4, 3], [1, 2, 3, 4]);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// verdict, d_row, d_col, d_min, preuve
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

let (verdict_a, _dr_a, _dc_a, _dm_a, preuve_a) = certifier_impossibilite(IDENTITE, RENVERS, 5, &dtab, &sidx);
println!("CAS A (antipode, k_max=5) : {} - {}", verdict_a, preuve_a);

let (verdict_b, _dr_b, _dc_b, _dm_b, preuve_b) = certifier_impossibilite(IDENTITE, H_B, 3, &dtab, &sidx);
println!("CAS B (row identique, k_max=3) : {} - {}", verdict_b, preuve_b);

let (verdict_c, _dr_c, _dc_c, _dm_c, preuve_c) = certifier_impossibilite(IDENTITE, H_C, 1, &dtab, &sidx);
println!("CAS C (frontiere, k_max=1) : {} - {}", verdict_c, preuve_c);

CAS A (antipode, k_max=5) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 6 + 6 = 12 > 5


CAS B (row identique, k_max=3) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 0 + 4 = 4 > 3


CAS C (frontiere, k_max=1) : POSSIBLE - d_perm(row) + d_perm(col) = 1 + 0 = 1 <= 1


## 4. Verification BFS tronque

Pour chaque cas IMPOSSIBLE, on verifie par BFS tronque que `H` n'est **jamais** dans les sommets a distance `<= k_max` de `G`. Le BFS tronque collecte tous les sommats atteignables en au plus `k_max` pas, sans reconstruire le graphe entier — c'est un outil de verification operationnelle, distinct du certificat analytique et utile comme contre-controle.

In [5]:
// === Section 4.1 : BFS tronque ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
const RENVERS: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
const H_B: Jeu = ([1, 2, 3, 4], [3, 4, 1, 2]);
const H_C: Jeu = ([1, 2, 4, 3], [1, 2, 3, 4]);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

#[allow(dead_code)]
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

// BFS tronque : tous les sommets a <= pmax pas de depart
fn bfs_tronque(depart: Jeu, pmax: usize) -> BTreeSet<Jeu> {
    let mut atteints: BTreeSet<Jeu> = BTreeSet::new();
    atteints.insert(depart);
    let mut frontiere: Vec<Jeu> = vec![depart];
    for _ in 0..pmax {
        let mut nouvelle: Vec<Jeu> = Vec::new();
        for &u in frontiere.iter() {
            for v in adj_chambre(u) {
                if !atteints.contains(&v) {
                    atteints.insert(v);
                    nouvelle.push(v);
                }
            }
        }
        frontiere = nouvelle;
        if frontiere.is_empty() {
            break;
        }
    }
    atteints
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

let atte_a: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 5);
let verif_a: bool = !atte_a.contains(&RENVERS);
println!("CAS A : sommets a <= 5 pas = {}, RENVERS present ? {}", atte_a.len(), py_bool(atte_a.contains(&RENVERS)));
println!("  Temoin IMPOSSIBLE verifie : {}", py_bool(verif_a));

let atte_b: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 3);
let verif_b: bool = !atte_b.contains(&H_B);
println!("CAS B : sommets a <= 3 pas = {}, H_B present ? {}", atte_b.len(), py_bool(atte_b.contains(&H_B)));
println!("  Temoin IMPOSSIBLE verifie : {}", py_bool(verif_b));

let atte_c: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 1);
let verif_c: bool = atte_c.contains(&H_C);
println!("CAS C : sommets a <= 1 pas = {}, H_C present ? {}", atte_c.len(), py_bool(atte_c.contains(&H_C)));
println!("  Temoin POSSIBLE verifie : {}", py_bool(verif_c));

assert!(verif_a && verif_b && verif_c);
println!("ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies");

CAS A : sommets a <= 5 pas = 235, RENVERS present ? False


  Temoin IMPOSSIBLE verifie : True


CAS B : sommets a <= 3 pas = 68, H_B present ? False


  Temoin IMPOSSIBLE verifie : True


CAS C : sommets a <= 1 pas = 7, H_C present ? True


  Temoin POSSIBLE verifie : True


ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies


## 5. Exemples guides -- le temoin en action

Trois exemples guident pas-a-pas l'utilisation du temoin sur des structures differentes des Cas A/B/C (section 3). Chaque exemple combine verification analytique (via la table `dtab`) et verification operationnelle (via BFS tronque) pour montrer que les deux voies sont d'accord.

### 5.1 Exemple guide -- Un autre cas IMPOSSIBLE

**Demarche** : on choisit une chambre `H` dont la structure est *differente* de l'antipode du Cas A — par exemple, `H` a la **ligne antipode** mais la **colonne identique** a `G`. Alors `d_perm(row_G, row_H) = 6` (antipode du permutaedre) et `d_perm(col_G, col_H) = 0` (meme colonne), donc `d_min = 6 + 0 = 6`. Pour `k_max = 3`, le verdict est `IMPOSSIBLE` puisque `6 > 3`.

**Verification operationnelle** par BFS tronque : on enumere tous les sommets atteignables en `k_max = 3` pas depuis `G`. Si `H` n'y figure pas, le verdict analytique est confirme operationnellement — c'est un **contre-controle** independant du certificat.

L'exemple ci-dessous execute cette verification et imprime les deux verifications (analytique + operationnelle) ; elles doivent etre d'accord.

In [6]:
// === Exemple guide 5.1 : un autre cas IMPOSSIBLE (row antipode, col identique) ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
// row antipode, col identique
const H_EXEMPLE: Jeu = ([4, 3, 2, 1], [1, 2, 3, 4]);
const K_MAX_EXEMPLE: u32 = 3; // 6 > 3 -> IMPOSSIBLE

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// verdict, d_row, d_col, d_min, preuve
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

// BFS tronque : tous les sommets a <= pmax pas de depart
fn bfs_tronque(depart: Jeu, pmax: usize) -> BTreeSet<Jeu> {
    let mut atteints: BTreeSet<Jeu> = BTreeSet::new();
    atteints.insert(depart);
    let mut frontiere: Vec<Jeu> = vec![depart];
    for _ in 0..pmax {
        let mut nouvelle: Vec<Jeu> = Vec::new();
        for &u in frontiere.iter() {
            for v in adj_chambre(u) {
                if !atteints.contains(&v) {
                    atteints.insert(v);
                    nouvelle.push(v);
                }
            }
        }
        frontiere = nouvelle;
        if frontiere.is_empty() {
            break;
        }
    }
    atteints
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// Verification analytique (via la table dtab)
let (verdict_ex, _dr, _dc, _dm, preuve_ex) =
    certifier_impossibilite(IDENTITE, H_EXEMPLE, K_MAX_EXEMPLE, &dtab, &sidx);
println!("Certificat analytique : {} - {}", verdict_ex, preuve_ex);

// Verification operationnelle (BFS tronque, independante de la table)
let atteints_ex: BTreeSet<Jeu> = bfs_tronque(IDENTITE, K_MAX_EXEMPLE as usize);
let h_present: bool = atteints_ex.contains(&H_EXEMPLE);
println!(
    "BFS tronque : {} sommets a <= {} pas",
    atteints_ex.len(),
    K_MAX_EXEMPLE
);
println!("H present dans le voisinage ? {} (attendu : False)", py_bool(h_present));
println!(
    "Verifications d'accord : {}",
    py_bool(verdict_ex == "IMPOSSIBLE" && !h_present)
);

Certificat analytique : IMPOSSIBLE - d_perm(row) + d_perm(col) = 6 + 0 = 6 > 3


BFS tronque : 68 sommets a <= 3 pas


H present dans le voisinage ? False (attendu : False)


Verifications d'accord : True


### 5.2 Exemple guide -- Distance d'un antipode

**Demarche** : la distance entre deux sommets `G` et `H` est le plus petit `k` tel que `H` appartient aux sommets a distance `<= k` de `G`. Pour l'antipode `(IDENTITE, RENVERS)`, le certificat analytique donne `d_min = 6 + 6 = 12` — c'est le diametre du graphe produit. Le plus petit `k_max` qui rend `POSSIBLE` est donc `k_max = 12` (a `k_max < 12`, IMPOSSIBLE est certain ; a `k_max >= 12`, POSSIBLE est garanti par concatenation des deux chemins optimaux).

L'exemple ci-dessous balaie les `k` de 0 a 12 et identifie le seuil de bascule.

In [7]:
// === Exemple guide 5.2 : distance de l'antipode (IDENTITE, RENVERS) ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
const RENVERS: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// verdict, d_row, d_col, d_min, preuve
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

let mut seuil: Option<u32> = None;
let mut d_min_antipode: u32 = 0;
for k in 0..=12u32 {
    let (verdict, _dr, _dc, d_min, _preuve) = certifier_impossibilite(IDENTITE, RENVERS, k, &dtab, &sidx);
    if verdict == "POSSIBLE" {
        seuil = Some(k);
        d_min_antipode = d_min;
        println!("Bascule IMPOSSIBLE -> POSSIBLE a k={} (d_min = {})", k, d_min);
        break;
    }
}
println!(
    "Seuil exact : {}, distance de l'antipode = {}",
    seuil.unwrap_or(0u32),
    d_min_antipode
);

Bascule IMPOSSIBLE -> POSSIBLE a k=12 (d_min = 12)


Seuil exact : 12, distance de l'antipode = 12


### 5.3 Exemple guide -- Paires `d_row=0, d_col=6`

**Demarche** : `d_row = 0` signifie que la ligne de `G` est identique a la ligne de `H` ; `d_col = 6` signifie que la colonne de `G` est l'antipode de la colonne de `H`. Ce sont donc les paires ou seul le cote colonne doit etre parcoure dans toute sa longueur.

Pour les trouver : on balaie toutes les chambres et on garde celles ou la ligne est inchangee et la colonne est antipode (par convention, l'antipode est `(4, 3, 2, 1)` a droite comme a gauche). Le balayage donne plusieurs solutions ; l'exemple en garde 3 et imprime leur verdict aux seuils `k_max=5` (IMPOSSIBLE) et `k_max=6` (frontiere, POSSIBLE).

In [8]:
// === Exemple guide 5.3 : trois paires d_row=0, d_col=6 ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// verdict, d_row, d_col, d_min, preuve
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

// chambres : produit row-major
let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// balayage row-major : on garde les paires d_row=0, d_col=6
let mut candidates: Vec<(Jeu, Jeu)> = Vec::new();
for &g in chambres.iter() {
    for &h in chambres.iter() {
        if g == h {
            continue;
        }
        if dtab[sidx[&g.0]][sidx[&h.0]] == 0 && dtab[sidx[&g.1]][sidx[&h.1]] == 6 {
            candidates.push((g, h));
            if candidates.len() >= 3 {
                break;
            }
        }
    }
    if candidates.len() >= 3 {
        break;
    }
}

for (i, &(g, h)) in candidates.iter().enumerate() {
    let (v5, _dr5, _dc5, _dm5, _p5) = certifier_impossibilite(g, h, 5, &dtab, &sidx);
    let (v6, _dr6, _dc6, _dm6, _p6) = certifier_impossibilite(g, h, 6, &dtab, &sidx);
    println!("Paire {} : k_max=5 -> {}, k_max=6 -> {}", i + 1, v5, v6);
};

Paire 1 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE


Paire 2 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE


Paire 3 : k_max=5 -> IMPOSSIBLE, k_max=6 -> POSSIBLE


## 6. Inventaire -- combien de triplets sont IMPOSSIBLE ?

Echantillon aleatoire de 100 chambres representatives (seed=42 pour la reproductibilite), balayage des `k_max ∈ {0..11}`. La proportion d'IMPOSSIBLE **decroît quand `k_max` augmente** (plus la borne est large, plus il est facile de tomber dans la zone POSSIBLE ; plus elle est serree, plus l'impossibilite est facile a prouver), puis s'annule au-dela du diametre (`k_max >= 12` : tout couple devient POSSIBLE).

La sortie de la cellule code plus loin trace les valeurs reelles sur le sous-echantillon :

| `k_max` | IMPOSSIBLE / Total | % |
|---:|---:|---:|
| 0 | 57 500 / 57 500 | 100,0 % |
| 1 | 56 900 / 57 500 | 99,0 % |
| 2 | 55 000 / 57 500 | 95,7 % |
| 3 | 50 800 / 57 500 | 88,3 % |
| 4 | 43 700 / 57 500 | 76,0 % |
| 5 | 34 100 / 57 500 | 59,3 % |

Ces valeurs sont celles qu'imprime la cellule code qui suit immediatement (cellule de substance preservee depuis le commit `6b190bffb0` ; seule la
reference de section de son commentaire d'en-tete a ete renumerotee,
`5.1` -> `6.1` -- ses sorties commitees n'ont pas bouge).

In [9]:
// === Section 6.1 : inventaire ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// MT19937 compatible CPython : random.Random(seed) avec seed entier >= 0 et < 2**32
// = init_by_array([seed]). Arithmetique u32 a debordement.
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

// ---- methode sample : algorithme CPython 3 de random.sample, ajoutee au bloc mt.rs
// (mt.rs lignes 1-75 = struct Mt19937 + new/new_by_array/next_u32/random/getrandbits/
//  randbelow/randint/choice/shuffle : a recopier tel quel, prouve dans les lots 1-2)
impl Mt19937 {
    fn sample<T: Clone>(&mut self, population: &[T], k: usize) -> Vec<T> {
        // precondition : k <= n (toujours vrai dans le carnet)
        let n = population.len();
        let mut result: Vec<T> = Vec::with_capacity(k);
        let mut setsize: u64 = 21;
        if k > 5 {
            // 4 ** ceil(log(3k, 4)) : plus petit 4^e >= 3k (entier exact)
            let mut p: u64 = 1;
            while p < 3 * k as u64 { p *= 4; }
            setsize += p;
        }
        if n as u64 <= setsize {
            // branche pool : invariant « non selectionnes dans pool[0..n-i] »
            let mut pool: Vec<T> = population.to_vec();
            for i in 0..k {
                let j = self.randbelow((n - i) as u64) as usize;
                result.push(pool[j].clone());
                pool[j] = pool[n - i - 1].clone();
            }
        } else {
            // branche ensemble de selectionnes (rejection des doublons)
            let mut selected: std::collections::BTreeSet<u64> = std::collections::BTreeSet::new();
            for _ in 0..k {
                let mut j = self.randbelow(n as u64);
                while selected.contains(&j) { j = self.randbelow(n as u64); }
                selected.insert(j);
                result.push(population[j as usize].clone());
            }
        }
        result
    }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

// population du tirage : chambres dans l'ordre row-major
let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// random.seed(42) ; random.sample(chambres, 100)
let mut rng = Mt19937::new(42);
let echantillon: Vec<Jeu> = rng.sample(&chambres, 100);

let mut resultats_par_k: Vec<(u32, u32)> = Vec::new();
for k_max in 0..12u32 {
    let mut compteur_imp: u32 = 0;
    let mut compteur_tot: u32 = 0;
    for &g in echantillon.iter() {
        for &h in chambres.iter() {
            if g == h {
                continue;
            }
            let d_min = dtab[sidx[&g.0]][sidx[&h.0]] + dtab[sidx[&g.1]][sidx[&h.1]];
            compteur_tot += 1;
            if d_min > k_max {
                compteur_imp += 1;
            }
        }
    }
    resultats_par_k.push((compteur_imp, compteur_tot));
}

println!("k_max | IMPOSSIBLE / Total | %");
println!("{}", "-".repeat(50));
for k_max in 0..12u32 {
    let (imp, tot) = resultats_par_k[k_max as usize];
    let pct = 100.0 * imp as f64 / tot as f64;
    println!("  {:>2}  | {:>6} / {:>6} | {:>5.1}%", k_max, imp, tot, pct);
}

let (imp5, tot5) = resultats_par_k[5];
println!();
println!(
    "A k_max=5 : {:.1}% des paires sont IMPOSSIBLE -- le temoin tranche souvent",
    100.0 * imp5 as f64 / tot5 as f64
);

k_max | IMPOSSIBLE / Total | %


--------------------------------------------------


   0  |  57500 /  57500 | 100.0%


   1  |  56900 /  57500 |  99.0%


   2  |  55000 /  57500 |  95.7%


   3  |  50800 /  57500 |  88.3%


... (5 outputs left out)

   9  |   2600 /  57500 |   4.5%


  10  |    700 /  57500 |   1.2%


  11  |    100 /  57500 |   0.2%


A k_max=5 : 59.3% des paires sont IMPOSSIBLE -- le temoin tranche souvent


## 7. Frontiere et bornes

**Certifie** : pour `k_max < 12`, IMPOSSIBLE est certain des que `d_min > k_max`.

**Non certifie** : un cas POSSIBLE garantit l'**existence** d'un chemin, pas un chemin explicite. La construction explicite est deléguée au constructeur de GT-20 (point 1 du critere #12205 §5) qui produit le chemin par concatenation des chemins optimaux dans chaque facteur.

**Au-dela de `k_max = 12`** (le diametre du produit), tout est POSSIBLE par definition : le chemin existe toujours en au plus 12 pas.

In [10]:
// === Section 7.1 : resume final ===
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);
const RENVERS: Jeu = ([4, 3, 2, 1], [4, 3, 2, 1]);
const H_B: Jeu = ([1, 2, 3, 4], [3, 4, 1, 2]);
const H_C: Jeu = ([1, 2, 4, 3], [1, 2, 3, 4]);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// verdict, d_row, d_col, d_min, preuve
fn certifier_impossibilite(
    g: Jeu,
    h: Jeu,
    k_max: u32,
    dtab: &Vec<Vec<u32>>,
    sidx: &HashMap<Perm, usize>,
) -> (&'static str, u32, u32, u32, String) {
    let d_row = dtab[sidx[&g.0]][sidx[&h.0]];
    let d_col = dtab[sidx[&g.1]][sidx[&h.1]];
    let d_min = d_row + d_col;
    let verdict: &'static str = if d_min > k_max { "IMPOSSIBLE" } else { "POSSIBLE" };
    let preuve = format!(
        "d_perm(row) + d_perm(col) = {} + {} = {} {} {}",
        d_row,
        d_col,
        d_min,
        if d_min > k_max { ">" } else { "<=" },
        k_max
    );
    (verdict, d_row, d_col, d_min, preuve)
}

// BFS tronque : tous les sommets a <= pmax pas de depart
fn bfs_tronque(depart: Jeu, pmax: usize) -> BTreeSet<Jeu> {
    let mut atteints: BTreeSet<Jeu> = BTreeSet::new();
    atteints.insert(depart);
    let mut frontiere: Vec<Jeu> = vec![depart];
    for _ in 0..pmax {
        let mut nouvelle: Vec<Jeu> = Vec::new();
        for &u in frontiere.iter() {
            for v in adj_chambre(u) {
                if !atteints.contains(&v) {
                    atteints.insert(v);
                    nouvelle.push(v);
                }
            }
        }
        frontiere = nouvelle;
        if frontiere.is_empty() {
            break;
        }
    }
    atteints
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// MT19937 compatible CPython : random.Random(seed) avec seed entier >= 0 et < 2**32
// = init_by_array([seed]). Arithmetique u32 a debordement.
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

// ---- methode sample : algorithme CPython 3 de random.sample, ajoutee au bloc mt.rs
// (mt.rs lignes 1-75 = struct Mt19937 + new/new_by_array/next_u32/random/getrandbits/
//  randbelow/randint/choice/shuffle : a recopier tel quel, prouve dans les lots 1-2)
impl Mt19937 {
    fn sample<T: Clone>(&mut self, population: &[T], k: usize) -> Vec<T> {
        // precondition : k <= n (toujours vrai dans le carnet)
        let n = population.len();
        let mut result: Vec<T> = Vec::with_capacity(k);
        let mut setsize: u64 = 21;
        if k > 5 {
            // 4 ** ceil(log(3k, 4)) : plus petit 4^e >= 3k (entier exact)
            let mut p: u64 = 1;
            while p < 3 * k as u64 { p *= 4; }
            setsize += p;
        }
        if n as u64 <= setsize {
            // branche pool : invariant « non selectionnes dans pool[0..n-i] »
            let mut pool: Vec<T> = population.to_vec();
            for i in 0..k {
                let j = self.randbelow((n - i) as u64) as usize;
                result.push(pool[j].clone());
                pool[j] = pool[n - i - 1].clone();
            }
        } else {
            // branche ensemble de selectionnes (rejection des doublons)
            let mut selected: std::collections::BTreeSet<u64> = std::collections::BTreeSet::new();
            for _ in 0..k {
                let mut j = self.randbelow(n as u64);
                while selected.contains(&j) { j = self.randbelow(n as u64); }
                selected.insert(j);
                result.push(population[j as usize].clone());
            }
        }
        result
    }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// structure produit : recomptage des ecarts sur les 576 chambres
let dist_chambre: HashMap<Jeu, u32> = bfs(IDENTITE, adj_chambre);
let mut ecarts: u32 = 0;
for &g in chambres.iter() {
    let (r, c) = g;
    let d_ch = *dist_chambre.get(&g).unwrap();
    let d_calc = *dist_perm[0].get(&r).unwrap() + *dist_perm[0].get(&c).unwrap();
    if d_ch != d_calc {
        ecarts += 1;
    }
}

// les trois cas fondateurs
let (verdict_a, _dra, _dca, _dma, _pa) = certifier_impossibilite(IDENTITE, RENVERS, 5, &dtab, &sidx);
let (verdict_b, _drb, _dcb, _dmb, _pb) = certifier_impossibilite(IDENTITE, H_B, 3, &dtab, &sidx);
let (verdict_c, _drc, _dcc, _dmc, _pc) = certifier_impossibilite(IDENTITE, H_C, 1, &dtab, &sidx);

// les trois BFS tronques
let atte_a: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 5);
let verif_a: bool = !atte_a.contains(&RENVERS);
let atte_b: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 3);
let verif_b: bool = !atte_b.contains(&H_B);
let atte_c: BTreeSet<Jeu> = bfs_tronque(IDENTITE, 1);
let verif_c: bool = atte_c.contains(&H_C);

// l'inventaire (meme tirage que la section 6)
let mut rng = Mt19937::new(42);
let echantillon: Vec<Jeu> = rng.sample(&chambres, 100);
let mut resultats_par_k: Vec<(u32, u32)> = Vec::new();
for k_max in 0..12u32 {
    let mut compteur_imp: u32 = 0;
    let mut compteur_tot: u32 = 0;
    for &g in echantillon.iter() {
        for &h in chambres.iter() {
            if g == h {
                continue;
            }
            let d_min = dtab[sidx[&g.0]][sidx[&h.0]] + dtab[sidx[&g.1]][sidx[&h.1]];
            compteur_tot += 1;
            if d_min > k_max {
                compteur_imp += 1;
            }
        }
    }
    resultats_par_k.push((compteur_imp, compteur_tot));
}
let (imp5, tot5) = resultats_par_k[5];

println!("RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth");
println!();
println!("Verdict par cas :");
println!("  CAS A : {}", verdict_a);
println!("  CAS B : {}", verdict_b);
println!("  CAS C : {}", verdict_c);
println!();
println!("Verification par BFS tronque :");
println!("  CAS A IMPOSSIBLE verifie : {}", py_bool(verif_a));
println!("  CAS B IMPOSSIBLE verifie : {}", py_bool(verif_b));
println!("  CAS C POSSIBLE verifie   : {}", py_bool(verif_c));
println!();
println!(
    "Structure produit : {}/{} chambres, {} ecart",
    chambres.len(),
    chambres.len(),
    ecarts
);
println!("Table 24x24 : symetrique et triangulaire");
println!(
    "A k_max=5 : {:.1}% des paires sont IMPOSSIBLE",
    100.0 * imp5 as f64 / tot5 as f64
);

RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth


Verdict par cas :


  CAS A : IMPOSSIBLE


  CAS B : IMPOSSIBLE


  CAS C : POSSIBLE


Verification par BFS tronque :


... (2 outputs left out)

  CAS C POSSIBLE verifie   : True


Structure produit : 576/576 chambres, 0 ecart


Table 24x24 : symetrique et triangulaire


A k_max=5 : 59.3% des paires sont IMPOSSIBLE


## 8. Exercices

### Exercice A -- Un cas IMPOSSIBLE non antipode

Trouver un couple `(G, H)` et un `k_max` tels que le verdict est `IMPOSSIBLE`, avec une **structure differente** des cas fondateurs suivants :

- **Cas A** : antipode integral (les deux cotes a distance 6) -- exhibe en cellule 8 ;
- **Cas B** (`d_row = 0, d_col = 4`) : un seul cote avec une distance **non antipode** (4 < 6) ; aussi exhibe en cellule 8 ;
- **Exemple 5.1** (`H_EXEMPLE` = `((4, 3, 2, 1), (1, 2, 3, 4))`) : un seul cote **antipode** (`d_perm(row_G, row_H) = 6`, `d_perm(col_G, col_H) = 0`) -- l'exemple guide de la section 5.

L'enonce de l'exercice exclut donc l'antipode integral (Cas A) **et** les mono-cotes antipodes (exemple 5.1) **et** les mono-cotes non antipodes (Cas B). Reste ce qui n'appartient a aucune de ces trois familles : par exemple, les deux cotes a des distances intermediaires (3 et 4, par exemple) pour un `k_max = 5`. Verifier par BFS tronque que le verdict certifie est bien `IMPOSSIBLE`.

In [11]:
// Exercice A : cas IMPOSSIBLE non antipode
type Perm = [u8; 4];
type Jeu = (Perm, Perm);

// A l'etudiant : choisir G, H, k_max de sorte que d_row + d_col > k_max
// avec une structure differente des Cas A/B/5.1.
let _g_a: Option<Jeu> = None; // TODO etudiant
let _h_a: Option<Jeu> = None; // TODO etudiant
let _k_max_a: Option<u32> = None; // TODO etudiant
let _resultat_a: Option<&str> = None; // TODO etudiant

// Verification par l'etudiant :
// 1. certifier_impossibilite(G_A, H_A, k_max_A, &dtab, &sidx)
// 2. bfs_tronque(G_A, k_max_A as usize)
// 3. assert verdict == "IMPOSSIBLE" && H_A not in atteints

println!("Exercice A : a completer (cas IMPOSSIBLE non antipode)");

Exercice A : a completer (cas IMPOSSIBLE non antipode)


### Exercice B -- Structure produit pour un G arbitraire

Verifier que la structure produit reste valide pour un `G` de depart qui n'est **pas** `IDENTITE` — par exemple `G = ((2, 1, 4, 3), (3, 4, 1, 2))`. La structure produit est-elle toujours `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`, ou faut-il la reformuler ?

In [12]:
// Exercice B : verifier la structure produit pour un G != IDENTITE
type Perm = [u8; 4];
type Jeu = (Perm, Perm);

let g_b: Jeu = ([2, 1, 4, 3], [3, 4, 1, 2]); // un G arbitraire (non IDENTITE)

// A l'etudiant :
// 1. Calculer dist_chambre_G = bfs(g_b, adj_chambre)
// 2. Pour chaque H in chambres, comparer dist_chambre_G[&H] a
//    dtab[sidx[&g_b.0]][sidx[&H.0]] + dtab[sidx[&g_b.1]][sidx[&H.1]]
// 3. Conclure : la structure produit tient-elle ? Pourquoi ?
let _ecarts_b: Option<u32> = None; // TODO etudiant
let _resultat_b: Option<&str> = None; // TODO etudiant

println!("Exercice B : a completer (structure produit pour un G arbitraire)");

Exercice B : a completer (structure produit pour un G arbitraire)


### Exercice C -- Densite d'IMPOSSIBLE a k_max=7 par echantillonnage stratifie

La section 6 a montré une densite a `k_max=5` sur 100 chambres aleatoires. Estimer la densite a `k_max=7` en utilisant un echantillon **stratifie** : le scaffold ci-dessous partitionne le permutaedre en **7 strates** selon la distance a `IDENTITE` (0 a 6) et tire jusqu'a **4 ordres stricts `Perm` par strate**, soit 20 ordres au total. Ces `Perm` representent les lignes des chambres `Jeu` a comparer : chaque ligne tiree `t` est completee avec chacune des 24 colonnes `c` pour former les chambres `(t, c)`. On construit ensuite la population complete `chambres` dans l'ordre row-major, puis on compare chaque chambre de l'echantillon a chaque chambre de `chambres`, en evitant les cas `G == H`. Completer le comptage de la proportion d'IMPOSSIBLE, puis comparer au chiffre donne par la section 6 (qui utilise un echantillon aleatoire non stratifie).

In [13]:
// Exercice C : densite IMPOSSIBLE a k_max=7 par echantillonnage stratifie
use std::collections::{BTreeSet, HashMap, VecDeque};

type Perm = [u8; 4];
type Jeu = (Perm, Perm);

const IDENTITE: Jeu = ([1, 2, 3, 4], [1, 2, 3, 4]);

fn swap_valeurs_adjacentes(t: Perm, k: u8) -> Perm {
    let mut l = t;
    let mut pk: usize = 0;
    let mut pk1: usize = 0;
    for (i, &v) in l.iter().enumerate() {
        if v == k { pk = i; }
        if v == k + 1 { pk1 = i; }
    }
    l.swap(pk, pk1);
    l
}

fn swap_jeu(jeu: Jeu, cote: &str, k: u8) -> Jeu {
    let (row, col) = jeu;
    if cote == "ligne" {
        (swap_valeurs_adjacentes(row, k), col)
    } else {
        (row, swap_valeurs_adjacentes(col, k))
    }
}

fn adj_chambre(g: Jeu) -> Vec<Jeu> {
    // ordre prescrit : ligne k=1,2,3 puis colonne k=1,2,3
    let mut out: Vec<Jeu> = Vec::new();
    for cote in ["ligne", "colonne"] {
        for k in 1u8..=3 {
            out.push(swap_jeu(g, cote, k));
        }
    }
    out
}

fn adj_perm(t: Perm) -> [Perm; 3] {
    [
        swap_valeurs_adjacentes(t, 1),
        swap_valeurs_adjacentes(t, 2),
        swap_valeurs_adjacentes(t, 3),
    ]
}

fn bfs<T: Copy + Eq + std::hash::Hash, F: Fn(T) -> Vec<T>>(depart: T, voisins_fn: F) -> HashMap<T, u32> {
    let mut dist: HashMap<T, u32> = HashMap::new();
    dist.insert(depart, 0u32);
    let mut q: VecDeque<T> = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = *dist.get(&u).unwrap();
        for v in voisins_fn(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn gen_perm(prefix: &mut [u8; 4], pos: usize, used: &mut [bool; 4], out: &mut Vec<Perm>) {
    if pos == 4 {
        out.push(*prefix);
        return;
    }
    for v in 0..4usize {
        if !used[v] {
            used[v] = true;
            prefix[pos] = (v + 1) as u8;
            gen_perm(prefix, pos + 1, used, out);
            used[v] = false;
        }
    }
}

// MT19937 compatible CPython : random.Random(seed) avec seed entier >= 0 et < 2**32
// = init_by_array([seed]). Arithmetique u32 a debordement.
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64 ; mots petit-boutistes, le dernier tronque
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

// ---- methode sample : algorithme CPython 3 de random.sample, ajoutee au bloc mt.rs
// (mt.rs lignes 1-75 = struct Mt19937 + new/new_by_array/next_u32/random/getrandbits/
//  randbelow/randint/choice/shuffle : a recopier tel quel, prouve dans les lots 1-2)
impl Mt19937 {
    fn sample<T: Clone>(&mut self, population: &[T], k: usize) -> Vec<T> {
        // precondition : k <= n (toujours vrai dans le carnet)
        let n = population.len();
        let mut result: Vec<T> = Vec::with_capacity(k);
        let mut setsize: u64 = 21;
        if k > 5 {
            // 4 ** ceil(log(3k, 4)) : plus petit 4^e >= 3k (entier exact)
            let mut p: u64 = 1;
            while p < 3 * k as u64 { p *= 4; }
            setsize += p;
        }
        if n as u64 <= setsize {
            // branche pool : invariant « non selectionnes dans pool[0..n-i] »
            let mut pool: Vec<T> = population.to_vec();
            for i in 0..k {
                let j = self.randbelow((n - i) as u64) as usize;
                result.push(pool[j].clone());
                pool[j] = pool[n - i - 1].clone();
            }
        } else {
            // branche ensemble de selectionnes (rejection des doublons)
            let mut selected: std::collections::BTreeSet<u64> = std::collections::BTreeSet::new();
            for _ in 0..k {
                let mut j = self.randbelow(n as u64);
                while selected.contains(&j) { j = self.randbelow(n as u64); }
                selected.insert(j);
                result.push(population[j as usize].clone());
            }
        }
        result
    }
}

let mut toutes: Vec<Perm> = Vec::new();
let mut prefix: [u8; 4] = [0, 0, 0, 0];
let mut used: [bool; 4] = [false, false, false, false];
gen_perm(&mut prefix, 0, &mut used, &mut toutes);
let mut ensemble: BTreeSet<Perm> = BTreeSet::new();
for p in toutes {
    ensemble.insert(p);
}
let stricts: Vec<Perm> = ensemble.into_iter().collect();

// Population complete des chambres, dans l'ordre row-major.
let mut chambres: Vec<Jeu> = Vec::new();
for &r in stricts.iter() {
    for &c in stricts.iter() {
        chambres.push((r, c));
    }
}

let mut sidx: HashMap<Perm, usize> = HashMap::new();
for (i, &p) in stricts.iter().enumerate() {
    sidx.insert(p, i);
}

let dist_perm: Vec<HashMap<Perm, u32>> = stricts
    .iter()
    .map(|&o| bfs(o, |x: Perm| adj_perm(x).to_vec()))
    .collect();
let mut dtab: Vec<Vec<u32>> = Vec::with_capacity(stricts.len());
for i in 0..stricts.len() {
    let mut ligne: Vec<u32> = Vec::with_capacity(stricts.len());
    for j in 0..stricts.len() {
        ligne.push(*dist_perm[i].get(&stricts[j]).unwrap());
    }
    dtab.push(ligne);
}

// graine differente de la section 6 (seed=42)
let mut rng = Mt19937::new(2025);

// Stratification par distance a IDENTITE dans le permutaedre
let mut stricts_par_dist: Vec<Vec<Perm>> = vec![Vec::new(); 7];
for &t in stricts.iter() {
    let d = dtab[sidx[&IDENTITE.0]][sidx[&t]] as usize;
    if d < 7 {
        stricts_par_dist[d].push(t);
    }
}

// 7 appels consecutifs sur la meme instance (20 ordres au total)
let mut echantillon_stratifie: Vec<Perm> = Vec::new();
for d in 0..7usize {
    let k = 4.min(stricts_par_dist[d].len());
    let s = rng.sample(&stricts_par_dist[d], k);
    echantillon_stratifie.extend(s);
}

// Chaque Perm tiree est une ligne. Pour obtenir des chambres completes,
// on l'associate a chacune des 24 colonnes : G = (t, c).
let mut echantillon_jeux: Vec<Jeu> = Vec::new();
for &row in echantillon_stratifie.iter() {
    for &col in stricts.iter() {
        echantillon_jeux.push((row, col));
    }
}

// A l'etudiant :
// 1. Pour chaque G dans echantillon_jeux, pour chaque H dans chambres,
//    compter les IMPOSSIBLE a k_max=7 (en evitant G == H)
// 2. Calculer d_min avec dtab[sidx[&G.0]][sidx[&H.0]]
//    + dtab[sidx[&G.1]][sidx[&H.1]]
// 3. Comparer au chiffre aleatoire de la section 6
let _resultat_c: Option<f64> = None; // TODO etudiant

println!("Exercice C : a completer (densite IMPOSSIBLE a k_max=7, echantillon stratifie)");

Exercice C : a completer (densite IMPOSSIBLE a k_max=7, echantillon stratifie)


## Conclusion -- le point 4 sur Robinson-Goforth est tenu

**Acceptance #12205 §5** :

1. Substrat dont le verificateur existe sur main : GT-20 porte les points 1-3. **OUI**.
2. Generateur != verificateur : `certifier_impossibilite` (O(1)) != BFS exhaustif (O(V + E) sur la boule visitée, ici O(576 + 3 456) par palier). **OUI**.
3. Temoin certifie : verdict IMPOSSIBLE porte certificat analytique (d_row, d_col, k_max). **OUI**.
4. Cas sans solution = temoin d'impossibilite, pas silence : section 4 verifie par BFS tronque. Section 6 : plus de la moitie des paires a k_max=5 sont IMPOSSIBLE. **OUI**.

**Structure du notebook** :
- Sections 1-2 : preuve analytique de la structure produit (borne inf + borne sup) ;
- Section 3 : 3 cas fondateurs A/B/C ;
- Section 4 : verification operationnelle par BFS tronque ;
- Section 5 : 3 exemples guides integres a la narration (5.1, 5.2, 5.3) ;
- Sections 6-7 : inventaire, frontiere et bornes ;
- Section 8 : 3 exercices non corriges (A, B, C) avec stubs `TODO etudiant`.

**Substrats independants portant les 4 points** :
- Life/Conway (PR #14205, #12395)
- AMD (`impossibility_strict_payments`)
- Robinson-Goforth (ce notebook)
- Tweety-5d (`afB_no_stable`, PR #13628)

Reference croisee : #12205 - #12364 - #13628.

## À retenir

- **Un cas sans solution est un *témoin d'impossibilité*, pas un silence.** Le critère #12205 §5 est tenu sur Robinson-Goforth : chaque verdict `IMPOSSIBLE` porte un **certificat analytique** `(d_row, d_col, k_max)` produit par `certifier_impossibilite` (O(1)), distinct du BFS exhaustif qui le *vérifie*.
- **La structure produit est le levier analytique** : `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`. Elle réduit un problème de 576 sommets à deux problèmes de 24, et rend le certificat trivial à produire.
- **L'impossibilité est fréquente, pas marginale** : à `k_max = 5`, plus de la moitié des paires de chambres sont `IMPOSSIBLE` (mesuré `59.3%`). Le témoin n'est pas un artefact du choix des cas.

## Vérifiez votre compréhension

1. Pourquoi le générateur (`certifier_impossibilite`, O(1)) et le vérificateur (BFS exhaustif) doivent-ils être *différents* ? Que certifie chacun ?
2. En quoi la structure produit `d_row + d_col` rend-elle le certificat d'impossibilité trivial à produire ?
3. Pourquoi le chiffre `59.3%` de paires impossibles à `k_max = 5` renforce-t-il la valeur du témoin plutôt qu'elle ne la limite ?

<details>
<summary>Éléments de réponse</summary>

1. Parce que les deux instruments ne certifient pas la même chose, et que l'un ne peut pas contrôler l'autre s'il partage son code. `certifier_impossibilite` lit deux entrées de la table 24×24 et additionne : il certifie **analytiquement** que `d_min = d_row + d_col > k_max`, donc qu'aucun chemin de longueur `<= k_max` n'existe — la preuve imprimée est l'inégalité elle-même : `d_perm(row) + d_perm(col) = 6 + 6 = 12 > 5` (cas A), `0 + 4 = 4 > 3` (cas B), `1 + 0 = 1 <= 1` (cas C). Le BFS tronqué ne lit ni le certificat ni la table : il explore le graphe réel depuis G et constate que H n'est pas dans la boule — `sommets a <= 5 pas = 235, RENVERS present ? False`, `68 ... False`, `7 ... True`. Si le même instrument produisait et vérifierait, une erreur de la table ou du théorème de structure passerait des deux côtés ; séparés, les deux voies doivent s'accorder, et elles s'accordent sur les trois cas fondateurs (`ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies`) comme sur le quatrième de l'exemple 5.1 (`6 + 0 = 6 > 3`, H absent de 68 sommets). Le générateur certifie une **borne** ; le vérificateur certifie une **inatteignabilité opérationnelle**.
2. Parce qu'elle ramène une distance sur `576` chambres à deux distances sur `24` ordres. Le théorème `d_chambre(G,H) = d_perm(row_G,row_H) + d_perm(col_G,col_H)` est vérifié sur `576/576` chambres avec `0 ecart` ; on peut donc précalculer une fois la table 24×24 — elle-même contrôlée, `0 asymetrie` et `0 violation` de l'inégalité triangulaire sur `13824` triplets — après quoi le certificat d'un triplet quelconque n'est plus qu'une somme de deux lectures de table, en O(1), au lieu d'un parcours des 576 sommets. Le diamètre se lit de la même façon : `Diametre permutaedre : 6` et `Diametre chambres : 12`, et l'antipode bascule de IMPOSSIBLE à POSSIBLE exactement à `k=12`. Le certificat n'est pas une approximation : il est exact pour toute paire par le théorème, et trivial à produire parce que deux entrées de table suffisent.
3. Parce qu'un témoin qui ne vaudrait que pour trois cas choisis serait anecdotique. L'inventaire de la section 6 montre que l'impossibilité est le régime majoritaire à borne serrée : à `k_max = 5`, la sortie donne `34100 / 57500`, soit `59.3%` des paires `IMPOSSIBLE`. Et la décroissance suit exactement ce que prédit le théorème : `100.0%` à `0`, `99.0%` à `1`, `95.7%` à `2`, `88.3%` à `3`, `76.0%` à `4`, `59.3%` à `5`, puis `0.2%` à `11`, la bascule IMPOSSIBLE → POSSIBLE tombant à `k=12`, le diamètre. Le témoin n'est donc pas un cas limite fabriqué : c'est l'instrument qui répond, avec un certificat, dans le régime où la question se pose le plus souvent — et le fait que la fréquence décroisse puis s'annule au diamètre confirme que le verdict dépend de la borne, pas d'une bizarrerie des cas exhibés.

</details>

## Pour aller plus loin

- **Les substrats indépendants qui portent les 4 points du critère** : Life/Conway, AMD (`impossibility_strict_payments`), Tweety-5d (`afB_no_stable`).
- **L'échantillonnage stratifié** (exercice C) : estimer une densité sur un permutaèdre en partitionnant par distance à l'identité.
- Le lien avec les témoins d'impossibilité en théorie des jeux bayésiens.